In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
raw = Path("../data/raw")

for f in sorted(raw.glob("*.csv")):
    df = pd.read_csv(f, nrows=5)
    n_rows = sum(1 for _ in open(f)) - 1
    print(f"\n=== {f.name} | rows ~ {n_rows:,} ===")
    print(df.dtypes)
    print(df.head(3))

In [ ]:
pat = pd.read_csv(raw/"patients.csv")
out = pd.read_csv(raw/"outcomes.csv", parse_dates=["admission_date","discharge_date"])

print("rows:", len(out), "| unique patients:", out.patient_id.nunique())
print("admissions per patient:\n", out.patient_id.value_counts().value_counts().sort_index())
print("patients matched:", out.patient_id.isin(pat.patient_id).mean())
print("admission range:", out.admission_date.min(), "->", out.admission_date.max())
print("LOS mismatch (max):", ((out.discharge_date - out.admission_date).dt.days - out.length_of_stay_days).abs().max())

print("\n--- disposition ---")
print(out.discharge_disposition.value_counts(dropna=False))
print(out.discharge_disposition.value_counts(normalize=True).round(3))

print("\n--- binary flags ---")
print(out[["icu_admission","in_hospital_death","readmitted_30d"]].mean().round(3))
print(pd.crosstab(out.discharge_disposition, out.in_hospital_death))
print(pd.crosstab(out.discharge_disposition, out.readmitted_30d, normalize="index").round(3))

print("\n--- signal check ---")
m = out.merge(pat, on="patient_id", how="left")
cols = ["age","charlson_index","systolic_bp","heart_rate","bmi","length_of_stay_days"]
print(m.groupby("discharge_disposition")[cols].mean().round(2))
print(m.groupby("readmitted_30d")[cols].mean().round(2))

In [ ]:
lab = pd.read_csv(raw/"lab_results.csv", usecols=["patient_id","test_date","test_name","value"], parse_dates=["test_date"])
l = lab.merge(out[["patient_id","admission_date","discharge_date"]], on="patient_id")
l["d_adm"] = (l.test_date - l.admission_date).dt.days
print("before admission:", round((l.d_adm < 0).mean(), 3))
print("during stay:", round(((l.test_date >= l.admission_date) & (l.test_date <= l.discharge_date)).mean(), 3))
print("after discharge:", round((l.test_date > l.discharge_date).mean(), 3))
print("labs per admitted patient:\n", l.groupby("patient_id").size().describe())
print("labs within 90d before admission:", ((l.d_adm < 0) & (l.d_adm >= -90)).mean().round(3))

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

feat = [c for c in pat.columns if c != "patient_id"]
X = m[feat].copy()
for c in X.select_dtypes(exclude="number").columns:
    X[c] = X[c].astype("category")

d = m.discharge_disposition
targets = {
    "home vs non-home (all)":        ((d != "home").astype(int), d.notna()),
    "home vs non-home (excl expired)": ((d != "home").astype(int), d != "expired"),
    "readmitted_30d":                (m.readmitted_30d, d != "expired"),
    "in_hospital_death":             (m.in_hospital_death, d.notna()),
    "icu_admission":                 (m.icu_admission, d.notna()),
}

cv = StratifiedKFold(5, shuffle=True, random_state=42)
model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      enable_categorical=True, eval_metric="logloss",
                      random_state=42, n_jobs=-1)

for name, (y, mask) in targets.items():
    s = cross_val_score(model, X[mask], y[mask], cv=cv, scoring="roc_auc")
    print(f"{name:35s} n={mask.sum():5d} pos={y[mask].sum():5d}  AUC={s.mean():.3f} ± {s.std():.3f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Xn = pd.get_dummies(m[feat], drop_first=True).astype(float)
mask = d != "expired"
y = m.readmitted_30d[mask]
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
s = cross_val_score(lr, Xn[mask], y, cv=cv, scoring="roc_auc")
print(f"LogReg readmission AUC = {s.mean():.3f} ± {s.std():.3f}")

lr.fit(Xn[mask], y)
coef = pd.Series(lr[-1].coef_[0], index=Xn.columns).sort_values(key=abs, ascending=False)
print(coef.head(8).round(3))

In [ ]:
dx = pd.read_csv(raw/"diagnoses.csv", parse_dates=["visit_date"])
dx = dx.merge(out[["patient_id","admission_date"]], on="patient_id")
pre = dx[dx.visit_date < dx.admission_date]
agg = pre.groupby("patient_id").agg(
    n_prior_visits=("visit_date","size"),
    n_prior_emergency=("visit_type", lambda s: (s=="emergency").sum()),
).reset_index()
m2 = m.merge(agg, on="patient_id", how="left").fillna({"n_prior_visits":0,"n_prior_emergency":0})

X2 = pd.get_dummies(m2[feat + ["n_prior_visits","n_prior_emergency"]], drop_first=True).astype(float)
for name, y2, mk in [("home vs non-home", (m2.discharge_disposition!="home").astype(int), d.notna()),
                     ("readmitted_30d", m2.readmitted_30d, d!="expired")]:
    s = cross_val_score(lr, X2[mk], y2[mk], cv=cv, scoring="roc_auc")
    print(f"{name:20s} AUC = {s.mean():.3f} ± {s.std():.3f}")